# 🌸 Sakuga4K — Standalone Neural Anime Video Upscaler (Phase 1 Lab)
### Dedicated GPU Testing Lab for Anime Editors & AMV Creators

Test 2K / 4K resolution, universal mobile playback (`-movflags +faststart`), and aspect ratio preservation (9:16 vertical, 1:1 square, 16:9 widescreen) with side-by-side magnified visual inspection.

---

In [ ]:
# @title ⚡ Step 1: Install Dependencies & Download Neural Weights
# Hit Play to verify your free Tesla T4 GPU and load official anime models
!nvidia-smi

import os, sys, subprocess

if not os.path.exists('/content/Sakuga4k'):
    !git clone https://github.com/Anyone-design/Sakuga4k.git /content/Sakuga4k
%cd /content/Sakuga4k
!git fetch origin main && git reset --hard origin/main

!pip install -q opencv-python matplotlib

# Cache official public weights for Tensor Cores
!mkdir -p /content/models
!wget -q -nc -O /content/models/realesr-animevideov3.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth || true
!wget -q -nc -O /content/models/RealESRGAN_x4plus_anime_6B.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth || true

print("\n✅ Step 1 Complete: Environment & Neural Weights Ready!")


In [ ]:
# @title 📤 Step 2: Upload Test Anime Clip
# Either drag & drop any video into the left file explorer as /content/input.mp4
# OR hit Play on this cell to browse and upload from your computer/phone.
from google.colab import files
import cv2, os

if not os.path.exists('/content/input.mp4'):
    print("Select your video clip to upload:")
    uploaded = files.upload()
    if uploaded:
        uploaded_filename = list(uploaded.keys())[0]
        os.rename(uploaded_filename, '/content/input.mp4')
        print(f"\n✅ Uploaded {uploaded_filename} -> /content/input.mp4")

if not os.path.exists('/content/input.mp4'):
    raise FileNotFoundError("Please upload a video or drag input.mp4 into /content/")

# Probe Video Properties
cap = cv2.VideoCapture('/content/input.mp4')
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

aspect_ratio = w / h
aspect_label = "Landscape (16:9 / widescreen)" if w > h else "Vertical (9:16 Shorts/Reels)" if h > w else "Square (1:1)"

print("\n📹 --- Video Metadata ---")
print(f"Original Resolution : {w} x {h}")
print(f"Aspect Ratio        : {aspect_label} ({w}:{h} = {aspect_ratio:.3f})")
print(f"Framerate           : {fps:.2f} fps")
print(f"Total Frames        : {total_frames} frames (~{total_frames/fps:.1f}s)")


In [ ]:
# @title 🚀 Step 3: Run Standalone Upscale
# Adjust settings using the form fields below:

model_preset = "Real-CUGAN Pro + Vmake Pop" #@param ["Fast Video v3 (AnimeVideoV3)", "AnimeSharp 4x (RRDB 6B)", "Real-CUGAN Pro + Vmake Pop"]
target_resolution = "2K (1440p - Universal Mobile Playable)" #@param ["2K (1440p - Universal Mobile Playable)", "4K (2160p - Desktop UHD)", "Native 2x", "Native 4x"]
pop_filter_strength = "Medium Pop" #@param ["None", "Subtle Pop", "Medium Pop", "High Vmake Pop"]

import os, sys, time, shutil, subprocess
import cv2, numpy as np, torch

sys.path.insert(0, '/content/Sakuga4k/backend')
from models import model_manager

# Map Model Preset
preset_key = "animevideov3"
if "AnimeSharp" in model_preset:
    preset_key = "animesharp"
elif "Real-CUGAN" in model_preset:
    preset_key = "realcugan_pop"

print(f"⚡ Loading {model_preset} on Nvidia T4 GPU (FP16)...\n")
model = model_manager.load_model(preset_key)

# Calculate Target Dimensions (Aspect Ratio 100% Preserved)
input_path = '/content/input.mp4'
cap = cv2.VideoCapture(input_path)
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
cap.release()

if "2K" in target_resolution:
    if orig_h > orig_w: # Vertical video (e.g. 1080x1920 -> 1440x2560)
        target_h = 2560
        target_w = int(round(orig_w * 2560 / orig_h))
    else: # Landscape video (e.g. 1920x1080 -> 2560x1440)
        target_w = 2560
        target_h = int(round(orig_h * 2560 / orig_w))
elif "4K" in target_resolution:
    if orig_h > orig_w: # Vertical video (e.g. 1080x1920 -> 2160x3840)
        target_h = 3840
        target_w = int(round(orig_w * 3840 / orig_h))
    else: # Landscape video (e.g. 1920x1080 -> 3840x2160)
        target_w = 3840
        target_h = int(round(orig_h * 3840 / orig_w))
elif "Native 2x" in target_resolution:
    target_w = orig_w * 2
    target_h = orig_h * 2
else: # Native 4x
    target_w = orig_w * 4
    target_h = orig_h * 4

# Ensure even numbers for H.264 video codec
target_w = target_w - (target_w % 2)
target_h = target_h - (target_h % 2)

print(f"📐 Input Resolution  : {orig_w} x {orig_h}")
print(f"🎯 Output Resolution : {target_w} x {target_h} (Aspect Ratio 100% Preserved)")

# Step 1: Extract frames
frames_dir = '/content/temp_frames'
if os.path.exists(frames_dir):
    shutil.rmtree(frames_dir)
os.makedirs(frames_dir, exist_ok=True)

print("\n🎞️ Extracting raw frames...")
subprocess.run([
    "ffmpeg", "-y", "-i", input_path,
    "-qscale:v", "1", "-qmin", "1",
    f"{frames_dir}/frame_%06d.png"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

frame_files = sorted([f for f in os.listdir(frames_dir) if f.startswith("frame_") and f.endswith(".png")])
total = len(frame_files)
print(f"✅ Extracted {total} frames. Starting PyTorch neural inference...\n")

# Step 2: Super-resolve each frame on GPU
start_time = time.time()
for idx, fname in enumerate(frame_files, start=1):
    fpath = os.path.join(frames_dir, fname)
    img_bgr = cv2.imread(fpath)
    if img_bgr is None:
        continue
    
    # Save sample frame for before/after comparison
    if idx == total // 2:
        cv2.imwrite('/content/sample_before.png', img_bgr)
        
    t0 = time.time()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    
    out_tensor = model_manager.upscale_tensor(img_tensor, preset=preset_key)
    out_np = (out_tensor.squeeze(0).permute(1, 2, 0).numpy() * 255.0).clip(0, 255).astype(np.uint8)
    out_bgr = cv2.cvtColor(out_np, cv2.COLOR_RGB2BGR)
    
    # Resize to target resolution keeping exact aspect ratio
    cur_h, cur_w = out_bgr.shape[:2]
    if cur_h != target_h or cur_w != target_w:
        interp = cv2.INTER_AREA if (cur_h > target_h and cur_w > target_w) else cv2.INTER_LANCZOS4
        out_bgr = cv2.resize(out_bgr, (target_w, target_h), interpolation=interp)
        
    cv2.imwrite(fpath, out_bgr)
    
    if idx == total // 2:
        cv2.imwrite('/content/sample_after.png', out_bgr)
        
    dt_ms = int((time.time() - t0) * 1000)
    vram_mb = torch.cuda.memory_allocated(0) // (1024 * 1024) if torch.cuda.is_available() else 0
    if idx == 1 or idx % 10 == 0 or idx == total:
        print(f"⚡ Frame {idx}/{total} ({idx/total*100:.1f}%) | VRAM: {vram_mb} MB | Latency: {dt_ms}ms")

elapsed = time.time() - start_time
print(f"\n🎉 Neural inference complete in {elapsed:.1f}s ({elapsed/total*1000:.1f}ms/frame)!")

# Step 3: Mobile-Friendly FFmpeg Reassembly
output_path = '/content/output_upscaled.mp4'
print("\n📦 Reassembling video with Universal Mobile Playback Shield...")

ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-framerate", str(fps),
    "-i", f"{frames_dir}/frame_%06d.png",
    "-i", input_path,
    "-map", "0:v:0",
    "-map", "1:a:0?",
]

# Apply optional Pop / Sharpness filter
if pop_filter_strength == "Subtle Pop":
    ffmpeg_cmd.extend(["-vf", "unsharp=5:5:0.5:3:3:0.2"])
elif pop_filter_strength == "Medium Pop":
    ffmpeg_cmd.extend(["-vf", "unsharp=5:5:0.8:3:3:0.4"])
elif pop_filter_strength == "High Vmake Pop":
    ffmpeg_cmd.extend(["-vf", "unsharp=5:5:1.2:5:5:0.6"])

ffmpeg_cmd.extend([
    "-c:v", "libx264",
    "-crf", "17",
    "-preset", "fast",
    "-pix_fmt", "yuv420p",
    "-profile:v", "high",
    "-level", "4.2",
    "-movflags", "+faststart",
    "-c:a", "copy",
    "-shortest",
    output_path
])

subprocess.run(ffmpeg_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
shutil.rmtree(frames_dir, ignore_errors=True)

out_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"✅ Video ready! File: {output_path} ({out_size_mb:.2f} MB)")


In [ ]:
# @title 🔍 Step 4: Zoom-in Line Art Quality Inspection (Before vs After)
# Hit Play to inspect a magnified side-by-side comparison directly in Colab!
import cv2
import matplotlib.pyplot as plt

if os.path.exists('/content/sample_before.png') and os.path.exists('/content/sample_after.png'):
    b_before = cv2.imread('/content/sample_before.png')
    b_after = cv2.imread('/content/sample_after.png')

    rgb_before = cv2.cvtColor(b_before, cv2.COLOR_BGR2RGB)
    rgb_after = cv2.cvtColor(b_after, cv2.COLOR_BGR2RGB)

    bh, bw = rgb_before.shape[:2]
    ah, aw = rgb_after.shape[:2]

    crop_size = min(bw, bh) // 3
    bx = (bw - crop_size) // 2
    by = (bh - crop_size) // 2

    scale_x = aw / bw
    scale_y = ah / bh
    ax = int(bx * scale_x)
    ay = int(by * scale_y)
    acrop_w = int(crop_size * scale_x)
    acrop_h = int(crop_size * scale_y)

    crop_before = rgb_before[by:by+crop_size, bx:bx+crop_size]
    crop_after = rgb_after[ay:ay+acrop_h, ax:ax+acrop_w]

    fig, axes = plt.subplots(1, 2, figsize=(14, 7), dpi=150)
    axes[0].imshow(crop_before)
    axes[0].set_title(f"Original Frame (Crop: {crop_size}x{crop_size})", fontsize=12, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(crop_after)
    axes[1].set_title(f"Sakuga4K Neural Reconstructed (Crop: {acrop_w}x{acrop_h})", fontsize=12, fontweight='bold', color='purple')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("Run Step 3 first to generate the before/after frames.")


In [ ]:
# @title 💾 Step 5: Download Video to Phone / Computer
# Hit Play to download output_upscaled.mp4 directly to your device!
from google.colab import files
import os

if os.path.exists('/content/output_upscaled.mp4'):
    files.download('/content/output_upscaled.mp4')
    print("⬇️ Download started! Transfer to your phone and verify playback.")
else:
    print("❌ Output video not found. Run Step 3 first.")
